# 11 - USS Distance Selection & Validation  (Track B)

**Not part of the detector pipeline.** Adds a per-frame USS distance to the
existing labels so Track B can compute geometric height and run camera/USS
fusion.

Alignment (notebook 10) is solved. The open question this notebook answers is
**which distance signal actually tracks the labeled target**, and whether it is
trustworthy enough to divide by.

### What we already learned on the sample sequence

| source | frame coverage | range | note |
|---|---|---|---|
| MAP object buffer | **20/20** | ~370-3000 mm | ~5 m range - the usable one |
| PDC zones | 3/20 | ~510-1140 mm | short-range (~1.2 m) parking warning |

So the default source is the **object buffer**. Its remaining weakness: taking
the min across all live slots can switch between objects in multi-object
scenes (monotonicity < 1 is the tell). This notebook measures that per
sequence and keeps only sequences whose distance curve behaves like a real
approach.

### Three validations (all runnable without extra ground truth)

1. **Monotonic approach** - distance should fall as the car closes in.
2. **Event agreement** - the curve should pass near the Label V2 event
   distance (2000 mm) during the approach.
3. **Unit sanity** - values must land in a plausible 0.2-5 m band.

Plus a **distance-overlay video** per sample sequence: the annotated approach
with the USS distance printed on each frame - simultaneously a correctness
check (does distance shrink as the box grows?) and a presentation figure.

In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    assert (REPO_DIR / ".git").exists(), \
        "repo not found - run scripts/bootstrap_colab.sh first (set REPO_URL)"
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
    # canmatrix 1.3.0 (2026-10-01) breaks `import asammdf` (TypeError in
    # asammdf/blocks/types.py) - pin until asammdf supports it
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "asammdf==8.8.27",
                    "canmatrix==1.2"], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None
RAW_ROOT_OVERRIDE = None

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")
print(f"raw root  : {RAW_ROOT}")

In [ ]:
import csv
import json as _json
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from vision_uss_research.alignment.mf4 import (approach_monotonicity,
                                              camera_for_direction,
                                              frame_distances)
from vision_uss_research.csv_io import read_csv_tolerant, repair_csv
from vision_uss_research.labeling.boxes_io import load_boxes_tolerant
from vision_uss_research.runs import finish_run, new_run_id, resolve_run, start_run
from vision_uss_research.sequences import (driving_direction, frames_window_id,
                                           sequence_files, with_drive_retry)

SOURCE = "objbuff"        # "objbuff" (~5 m) or "pdc" (short range)
N_FRAMES = 20             # must match the labeling window used in 02/03
RATIO_RANGE = (0.30, 0.85)
MAX_SEQUENCES = 300       # 0 = every alignable sequence
NUM_WORKERS = 8
SEED = 0

# acceptance thresholds for "this sequence's distance curve is trustworthy"
MIN_COVERAGE = 0.60       # fraction of frames with a real reading
MIN_MONOTONICITY = 0.80   # fraction of steps that approach
PLAUSIBLE_MM = (150.0, 5000.0)
EVENT_DISTANCE_MM = 2000.0   # the Label V2 spec threshold
EVENT_TOLERANCE_MM = 600.0

RUN_DIR = ARTIFACTS / "alignment" / new_run_id("alignment", "distance")
RESULTS_CSV = RUN_DIR / "sequence_distance_quality.csv"
FRAMES_CSV = RUN_DIR / "frame_distances.csv"
RUN_DIR.mkdir(parents=True, exist_ok=True)

align_run = resolve_run(ARTIFACTS / "alignment", "alignment",
                        require_file="alignment_probe.csv")
assert align_run is not None, "run notebook 10 first (alignment EDA)"
probe = pd.read_csv(align_run / "alignment_probe.csv")
alignable = probe[probe["alignable"].fillna(0) == 1]["sequence_id"].astype(str).tolist()
print(f"alignment run : {align_run.name}")
print(f"alignable seqs: {len(alignable)}")

if MAX_SEQUENCES and MAX_SEQUENCES < len(alignable):
    rng = np.random.default_rng(SEED)
    alignable = sorted(rng.choice(alignable, MAX_SEQUENCES, replace=False).tolist())

SEQ_FIELDS = ["sequence_id", "camera", "driving_direction", "direction_assumed",
              "source", "coverage", "monotonicity", "min_mm", "max_mm",
              "passes_event_check", "plausible", "accepted", "error"]

# tolerant resume: repair a results CSV written by an earlier schema (the
# direction_assumed column was added), so mixed-width rows do not crash resume
# Resume, but RETRY errored sequences: an error (transient Drive I/O, or a bug
# fixed since the row was written, e.g. the old unknown_direction drop) is NOT
# "done". Only successful rows count as done; the CSV is rewritten to just those
# so retried sequences don't produce duplicate rows.
done = set()
if RESULTS_CSV.exists():
    repair_csv(RESULTS_CSV, SEQ_FIELDS)
    prev = read_csv_tolerant(RESULTS_CSV, SEQ_FIELDS)
    prev["error"] = prev["error"].fillna("")
    good = (prev[prev["error"] == ""]
            .drop_duplicates("sequence_id", keep="last"))
    n_retry = prev["sequence_id"].nunique() - good["sequence_id"].nunique()
    good.to_csv(RESULTS_CSV, index=False)      # drop errored rows before reprocessing
    done = set(good["sequence_id"].astype(str))
    print(f"resuming: {len(done)} succeeded, {n_retry} errored -> will retry")
todo = [s for s in alignable if s not in done]

start_run(RUN_DIR, config={"source": SOURCE, "n_frames": N_FRAMES,
                           "ratio_range": list(RATIO_RANGE),
                           "min_coverage": MIN_COVERAGE,
                           "min_monotonicity": MIN_MONOTONICITY},
          inputs=[f"alignment/{align_run.name}"], repo_root=REPO_DIR)
print(f"to process    : {len(todo)}")
print(f"output        : {RUN_DIR}")

## Class coverage of the alignable set (run this first)

De-risks Track B before the heavy distance run: does the alignable subset
actually contain the objects the experiments need?

- **Focal calibration** needs the KNOWN-HEIGHT classes (dummychild 125 cm,
  pole/isopole 108 cm) - if those are thin here, calibration is shaky.
- **Fusion** needs class coverage across low and high.

Joins the alignment probe (all sequences) to the inventory's per-sequence
object names. Reads only two CSVs already on Drive; starts no heavy work.

In [ ]:
import re
import yaml

# object class per sequence comes from the inventory the alignment run consumed
_align_meta = _json.loads((align_run / "run.json").read_text())
_inv_name = next((i.split("/", 1)[1] for i in _align_meta.get("inputs", [])
                  if i.startswith("inventory/")), None)
inv_dir = (ARTIFACTS / "inventory" / _inv_name) if _inv_name else resolve_run(
    ARTIFACTS / "inventory", "inventory", require_file="inventory_folders.csv",
    exclude_suffix="_smoke")
folders = pd.read_csv(inv_dir / "inventory_folders.csv")

# canonical object names (merge Bush/bush, Dummychild -Central/dummychild, ...)
# from the versioned mapping; falls back to lowercase-only if unavailable
def _norm(s):
    return re.sub(r"[^a-z0-9]+", "", str(s).lower())
_alias = {}
try:
    _m = yaml.safe_load((REPO_DIR / "configs" / "datasets"
                         / "label_v2_class_mapping.yaml").read_text())
    for canon, variants in (_m.get("canonical_objects") or {}).items():
        for v in [canon, *variants]:
            _alias[_norm(v)] = canon
except Exception:
    pass
def canonical(obj):
    return _alias.get(_norm(obj), str(obj).strip().lower())

KNOWN_HEIGHT_CM = {"dummychild": 125, "pole": 108}   # focal-calibration anchors

aligned_ids = set(probe[probe["alignable"].fillna(0) == 1]["sequence_id"].astype(str))
folders["is_aligned"] = folders["sequence_id"].astype(str).isin(aligned_ids)

# explode pipe-separated object names, drop the "empty" traversable marker
long = (folders.assign(obj=folders["label_v2_objects"].fillna("").str.split("|"))
        .explode("obj"))
long["obj"] = long["obj"].map(canonical)
long = long[(long["obj"] != "") & (long["obj"] != "empty")]
# ONE row per (object, sequence): a sequence listing an object twice must not
# inflate the aligned count past the total (the >100% bug)
long = long.drop_duplicates(["obj", "sequence_id"])

cov = (long.groupby("obj")
       .agg(total_seqs=("sequence_id", "nunique"),
            aligned_seqs=("is_aligned", "sum"))
       .assign(aligned_pct=lambda d: (100 * d["aligned_seqs"] / d["total_seqs"]).round(1))
       .sort_values("aligned_seqs", ascending=False))
assert (cov["aligned_seqs"] <= cov["total_seqs"]).all(), "aligned exceeds total!"
cov["known_height_cm"] = [KNOWN_HEIGHT_CM.get(o, "") for o in cov.index]
display(cov)

print("Focal-calibration anchors (need enough ALIGNED sequences):")
for obj, h in KNOWN_HEIGHT_CM.items():
    n = int(cov.loc[obj, "aligned_seqs"]) if obj in cov.index else 0
    flag = "OK" if n >= 20 else "THIN - calibration may be noisy"
    print(f"  {obj:<12} {h} cm : {n} aligned sequences   [{flag}]")

thin = cov[(cov["aligned_seqs"] < 10) & (cov["total_seqs"] >= 20)]
if len(thin):
    print("\nClasses well-represented overall but THIN after alignment "
          "(fusion on these will be weak):")
    display(thin[["total_seqs", "aligned_seqs", "aligned_pct"]])
cov.to_csv(RUN_DIR / "class_coverage_aligned.csv")

## Compute per-frame distances (parallel, resumable)

In [ ]:
def process(seq_id):
    seq_dir = RAW_ROOT / seq_id
    row = {"sequence_id": seq_id, "source": SOURCE, "error": "", "direction_assumed": 0}
    frames = []
    try:
        direction = with_drive_retry(driving_direction, seq_dir)
        row["driving_direction"] = direction
        found = with_drive_retry(sequence_files, seq_dir)
        # The object-buffer distance is camera-independent (min over all slots
        # around the whole car); the camera only sets the frame->MF4 clock. So a
        # missing direction tag is NOT a reason to drop the sequence - pick any
        # available camera. (This was 36% of the first run's "errors".)
        cam = camera_for_direction(direction)
        if cam is None or found.get(f"{cam}_video") is None:
            cam = ("rear" if found.get("rear_video") is not None
                   else "front" if found.get("front_video") is not None else None)
            row["direction_assumed"] = 1
        row["camera"] = cam or ""
        if cam is None:
            row["error"] = "no_camera_video"; return row, frames
        video = found[f"{cam}_video"]
        mf4 = next(iter(sorted(seq_dir.glob("*.mf4"))), None)
        if video is None or mf4 is None:
            row["error"] = "missing_video_or_mf4"; return row, frames

        cap = cv2.VideoCapture(str(video))
        fps = cap.get(cv2.CAP_PROP_FPS) or 1e-6
        dur = cap.get(cv2.CAP_PROP_FRAME_COUNT) / fps
        cap.release()
        ratios = np.linspace(RATIO_RANGE[0], RATIO_RANGE[1], N_FRAMES)
        pts = ratios * dur

        r = frame_distances(mf4, cam, pts, source=SOURCE)
        if r["error"]:
            row["error"] = r["error"]; return row, frames
        d = r["distance_mm"]
        valid = d[np.isfinite(d)]
        row["coverage"] = round(float(r["coverage"]), 3)
        row["monotonicity"] = (round(float(r["monotonicity"]), 3)
                               if r["monotonicity"] is not None else None)
        row["min_mm"] = round(float(valid.min()), 1) if valid.size else None
        row["max_mm"] = round(float(valid.max()), 1) if valid.size else None
        # validation 2: does the curve pass near the labeled event distance?
        row["passes_event_check"] = int(
            valid.size > 0 and
            np.min(np.abs(valid - EVENT_DISTANCE_MM)) <= EVENT_TOLERANCE_MM)
        # validation 3: plausible physical band
        row["plausible"] = int(
            valid.size > 0 and PLAUSIBLE_MM[0] <= valid.min()
            and valid.max() <= PLAUSIBLE_MM[1])
        row["accepted"] = int(
            row["coverage"] >= MIN_COVERAGE
            and (row["monotonicity"] or 0) >= MIN_MONOTONICITY
            and row["plausible"] == 1)
        for i, (ratio, dist) in enumerate(zip(ratios, d)):
            frames.append({"sequence_id": seq_id, "camera": cam, "frame_idx": i,
                           "frame_ratio": round(float(ratio), 4),
                           "distance_mm": (round(float(dist), 1)
                                           if np.isfinite(dist) else "")})
    except Exception as e:
        row["error"] = f"{type(e).__name__}:{e}"
    return row, frames

def append(path, rows, fields):
    if not rows:
        return
    header = not path.exists()
    with path.open("a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        if header:
            w.writeheader()
        w.writerows(rows)

if todo:
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
        futs = [pool.submit(process, s) for s in todo]
        for fut in tqdm(as_completed(futs), total=len(futs),
                        desc="Distances", unit="seq"):
            row, frames = fut.result()
            append(RESULTS_CSV, [row], SEQ_FIELDS)
            append(FRAMES_CSV, frames,
                   ["sequence_id", "camera", "frame_idx", "frame_ratio", "distance_mm"])

seqs = read_csv_tolerant(RESULTS_CSV, SEQ_FIELDS)
for _c in ["coverage", "monotonicity", "min_mm", "max_mm",
           "passes_event_check", "plausible", "accepted", "direction_assumed"]:
    seqs[_c] = pd.to_numeric(seqs[_c], errors="coerce")
print(f"processed: {len(seqs)} sequences")

## Validation verdict

In [ ]:
ok = seqs[seqs["error"].fillna("") == ""]
print(f"sequences processed        : {len(seqs)}")
print(f"  without error            : {len(ok)}")
if len(ok):
    print(f"  coverage >= {MIN_COVERAGE}          : "
          f"{int((ok['coverage'] >= MIN_COVERAGE).sum())}/{len(ok)}")
    print(f"  monotonicity >= {MIN_MONOTONICITY}     : "
          f"{int((ok['monotonicity'].fillna(0) >= MIN_MONOTONICITY).sum())}/{len(ok)}")
    print(f"  plausible range          : {int(ok['plausible'].sum())}/{len(ok)}")
    print(f"  passes 2000 mm event chk : {int(ok['passes_event_check'].sum())}/{len(ok)}")
    print(f"\n  ACCEPTED (usable for height/fusion): "
          f"{int(ok['accepted'].sum())}/{len(ok)} "
          f"({100*ok['accepted'].mean():.1f}%)")

if len(ok):
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
    axes[0].hist(ok["coverage"].dropna(), bins=20, color="steelblue")
    axes[0].axvline(MIN_COVERAGE, color="r", ls="--"); axes[0].set_title("frame coverage")
    axes[1].hist(ok["monotonicity"].dropna(), bins=20, color="seagreen")
    axes[1].axvline(MIN_MONOTONICITY, color="r", ls="--")
    axes[1].set_title("approach monotonicity")
    axes[2].hist(ok["min_mm"].dropna(), bins=30, color="darkorange")
    axes[2].set_title("closest distance per sequence (mm)")
    fig.tight_layout(); fig.savefig(RUN_DIR / "distance_quality.jpg", dpi=110)
    plt.show()

if seqs["error"].fillna("").ne("").any():
    display(seqs[seqs["error"].fillna("") != ""]["error"].str.slice(0, 50)
            .value_counts().to_frame("n"))

### Per-class acceptance (the real verdict) + error breakdown

In [ ]:
# Per-class acceptance - the verdict that actually matters. Track B needs the
# focal anchors (dummychild, pole) to accept well and a few hundred per class
# for fusion; a low OVERALL rate is fine if those classes are healthy. Also
# splits errors from quality rejections so transient Drive failures (recoverable
# on rerun) are not mistaken for bad data.
try:
    _cov = pd.read_csv(RUN_DIR / "class_coverage_aligned.csv")
    _obj_col = _cov.columns[0]
    seq_obj = {}  # sequence_id -> canonical object (from the coverage join's inventory)
    _folders = pd.read_csv(inv_dir / "inventory_folders.csv")
    for _, fr_ in _folders.iterrows():
        objs = [canonical(o) for o in str(fr_["label_v2_objects"]).split("|")]
        objs = [o for o in objs if o and o != "empty"]
        if objs:
            seq_obj[str(fr_["sequence_id"])] = objs[0]  # primary target
except Exception as e:
    seq_obj = {}
    print("per-class breakdown unavailable:", e)

if seq_obj:
    s = seqs.copy()
    s["obj"] = s["sequence_id"].astype(str).map(seq_obj)
    s["errored"] = s["error"].fillna("") != ""
    by = (s.groupby("obj")
          .agg(n=("sequence_id", "nunique"),
               errored=("errored", "sum"),
               accepted=("accepted", lambda a: int(pd.to_numeric(a, errors="coerce")
                                                    .fillna(0).sum())))
          .assign(clean=lambda d: d["n"] - d["errored"]))
    by["accept_of_clean_pct"] = (100 * by["accepted"] / by["clean"].clip(lower=1)).round(1)
    by = by.sort_values("accepted", ascending=False)
    display(by[["n", "errored", "clean", "accepted", "accept_of_clean_pct"]])

    print("Focal-calibration anchors:")
    for anchor in ["dummychild", "pole"]:
        if anchor in by.index:
            r = by.loc[anchor]
            print(f"  {anchor:<12}: {int(r['accepted'])} accepted "
                  f"({r['accept_of_clean_pct']}% of clean, {int(r['errored'])} errored)")
        else:
            print(f"  {anchor:<12}: not in this sample")

# error breakdown: transient (rerun recovers) vs structural
if seqs["error"].fillna("").ne("").any():
    print("\nerror types (open:* are transient Drive I/O - rerun to recover):")
    display(seqs[seqs["error"].fillna("") != ""]["error"].str.split(":").str[0]
            .value_counts().to_frame("n"))

## Per-sequence approach curves

A trustworthy curve falls smoothly. Curves that jump are the object-switching
failure mode (the min-over-slots caveat) - those sequences are rejected.

In [ ]:
fr = pd.read_csv(FRAMES_CSV)
fr["distance_mm"] = pd.to_numeric(fr["distance_mm"], errors="coerce")
acc = set(seqs[seqs["accepted"] == 1]["sequence_id"].astype(str))
rej = set(seqs[(seqs["accepted"] == 0) & (seqs["error"].fillna("") == "")]
          ["sequence_id"].astype(str))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), sharey=True)
for ax, group, title in [(axes[0], acc, "ACCEPTED"), (axes[1], rej, "rejected")]:
    for sid in list(group)[:40]:
        g = fr[fr["sequence_id"].astype(str) == sid].sort_values("frame_ratio")
        ax.plot(g["frame_ratio"], g["distance_mm"], alpha=0.5, lw=1)
    ax.set_title(f"{title} (n={len(group)}, showing <=40)")
    ax.set_xlabel("approach ratio"); ax.grid(alpha=0.3)
axes[0].set_ylabel("USS distance (mm)")
fig.tight_layout(); fig.savefig(RUN_DIR / "approach_curves.jpg", dpi=110)
plt.show()

## Distance-overlay video (validation + presentation artifact)

In [ ]:
CLIP_SEQUENCES = 3     # accepted sequences to render
CLIP_FPS = 6
WINDOW_ROOT = ARTIFACTS / "frames" / frames_window_id(N_FRAMES, RATIO_RANGE)
clip_dir = RUN_DIR / "clips"; clip_dir.mkdir(exist_ok=True)

def render(seq_id):
    row = seqs[seqs["sequence_id"].astype(str) == seq_id].iloc[0]
    cam = row["camera"]
    fdir = WINDOW_ROOT / seq_id / cam
    imgs = sorted(fdir.glob("*.jpg"))
    if not imgs:
        return None                       # frame cache not populated for this seq
    dist = (fr[fr["sequence_id"].astype(str) == seq_id]
            .sort_values("frame_idx")["distance_mm"].tolist())
    first = cv2.imread(str(imgs[0])); h, w = first.shape[:2]
    out = clip_dir / f"{seq_id}_{cam}_distance.mp4"
    vw = cv2.VideoWriter(str(out), cv2.VideoWriter_fourcc(*"mp4v"),
                         CLIP_FPS, (w, h))
    for i, ip in enumerate(imgs):
        img = cv2.imread(str(ip))
        d = dist[i] if i < len(dist) else float("nan")
        text = f"USS: {d/1000:.2f} m" if np.isfinite(d) else "USS: no reading"
        cv2.rectangle(img, (0, 0), (w, 40), (0, 0, 0), -1)
        cv2.putText(img, f"{seq_id[:26]}  {text}", (10, 28),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 255, 255), 2)
        vw.write(img)
    vw.release()
    return out

made = []
for sid in list(acc)[:CLIP_SEQUENCES]:
    p = render(sid)
    if p:
        made.append(p); print("clip:", p.name)
if not made:
    print("no clips rendered - the frame cache has no frames for these sequences "
          "(run notebook 02/03 extraction for them first)")

In [ ]:
summary = {
    "processed": int(len(seqs)),
    "errors": int(seqs["error"].fillna("").ne("").sum()),
    "accepted": int(seqs["accepted"].fillna(0).sum()),
    "accepted_pct": round(100 * float(seqs["accepted"].fillna(0).mean()), 1),
    "source": SOURCE,
}
finish_run(RUN_DIR, summary=summary)
print(_json.dumps(summary, indent=1))
print(f"\nper-frame distances : {FRAMES_CSV}")
print(f"sequence quality    : {RESULTS_CSV}")

## Next in Track B

`frame_distances.csv` is the deliverable: per (sequence, frame) USS distance in
mm, joinable to the label manifest on `sequence_id` + `frame_idx`.

1. **Focal calibration** - on ACCEPTED sequences of known-height classes
   (dummychild 125 cm, pole 108 cm), least-squares fit
   `focal = box_px_height * distance / real_height`. One constant per camera.
2. **Height** - `height_mm = box_px_height * distance_mm / focal`; validate on
   held-out known-height objects and report error bars.
3. **Fusion** - join detections to USS `ClassProbHigh` on the same frames and
   test whether camera + USS beats either alone in the ambiguous 10-30 cm band.

If the accepted fraction is low, the fix is target-slot selection: instead of
min-over-all-slots, follow one object id, or filter slots by the approach
corridor - the min-over-slots switching is the dominant rejection reason.